In [76]:
from langchain.chat_models import ChatOpenAI
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.graphs import Neo4jGraph
from langchain.chains import GraphCypherQAChain
import os


In [77]:
from dotenv import load_dotenv
load_dotenv()


True

In [78]:
text = """
Albert Einstein[a] (14 March 1879 – 18 April 1955) was a German-born theoretical physicist best known for developing the theory of relativity. Einstein also made important contributions to quantum theory.[1][5] His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation".[6] He received the 1921 Nobel Prize in Physics for "his services to theoretical physics, and especially for his discovery of the law of the photoelectric effect".[7]

Born in the German Empire, Einstein moved to Switzerland in 1895, forsaking his German citizenship (as a subject of the Kingdom of Württemberg)[note 1] the following year. In 1897, at the age of seventeen, he enrolled in the mathematics and physics teaching diploma program at the Swiss federal polytechnic school in Zurich, graduating in 1900. He acquired Swiss citizenship a year later, which he kept for the rest of his life, and afterwards secured a permanent position at the Swiss Patent Office in Bern. In 1905, he submitted a successful PhD dissertation to the University of Zurich. In 1914, he moved to Berlin to join the Prussian Academy of Sciences and the Humboldt University of Berlin, becoming director of the Kaiser Wilhelm Institute for Physics in 1917; he also became a German citizen again, this time as a subject of the Kingdom of Prussia.[note 1] In 1933, while Einstein was visiting the United States, Adolf Hitler came to power in Germany. Horrified by the Nazi persecution of his fellow Jews,[8] he decided to remain in the US, and was granted American citizenship in 1940.[9] On the eve of World War II, he endorsed a letter to President Franklin D. Roosevelt alerting him to the potential German nuclear weapons program and recommending that the US begin similar research, later carried out as the Manhattan Project.

In 1905, sometimes described as his annus mirabilis (miracle year), he published four groundbreaking papers.[10] In them, he outlined a theory of the photoelectric effect, explained Brownian motion, introduced his special theory of relativity, and demonstrated that if the special theory is correct, mass and energy are equivalent to each other. In 1915, he proposed a general theory of relativity that extended his system of mechanics to incorporate gravitation. A cosmological paper that he published the following year laid out the implications of general relativity for the modeling of the structure and evolution of the universe as a whole.[11][12] In 1917, Einstein wrote a paper which introduced the concepts of spontaneous emission and stimulated emission, the latter of which is the core mechanism behind the laser and maser, and which contained a trove of information that would be beneficial to developments in physics later on, such as quantum electrodynamics and quantum optics.[13]

In the middle part of his career, Einstein made important contributions to statistical mechanics and quantum theory. Especially notable was his work on the quantum physics of radiation, in which light consists of particles, subsequently called photons. With physicist Satyendra Nath Bose, he laid the groundwork for Bose–Einstein statistics. For much of the last phase of his academic life, Einstein worked on two endeavors that ultimately proved unsuccessful. First, he advocated against quantum theory's introduction of fundamental randomness into science's picture of the world, objecting that "God does not play dice".[14] Second, he attempted to devise a unified field theory by generalizing his geometric theory of gravitation to include electromagnetism. As a result, he became increasingly isolated from mainstream modern physics.
"""

In [79]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = splitter.create_documents([text])


In [86]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
llm = HuggingFaceEndpoint(
    # repo_id = "google/gemma-2-2b-it",
    repo_id = "mistralai/Mistral-7B-Instruct-v0.2",
    # repo_id="deepseek-ai/DeepSeek-R1-0528",

    # huggingfacehub_api_token="YOUR_HF_TOKEN",
    huggingfacehub_api_token="YOUR_HF_TOKEN",
    task = "text-generation"
)

model = ChatHuggingFace(llm = llm)


In [91]:
from langchain_community.chat_models import ChatOllama
llm = ChatOllama(
    model="llama3:8b",  # or "llama2", "gemma2"
    temperature=0
)
graph_transformer = LLMGraphTransformer(llm=llm)
graph_docs = graph_transformer.convert_to_graph_documents(docs)

# Inspect one example
print(graph_docs[0].nodes[:5])
print(graph_docs[0].relationships[:5])


[Node(id='quantum theory', type='Theory', properties={}), Node(id='Nobel Prize in Physics', type='Award', properties={}), Node(id='theory of relativity', type='Theory', properties={}), Node(id='Albert Einstein', type='Person', properties={}), Node(id='mass–energy equivalence formula E = mc2', type='Formula', properties={})]
[Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Node(id='theory of relativity', type='Theory', properties={}), type='DEVELOPED_THEORY_OF', properties={}), Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Node(id='quantum theory', type='Theory', properties={}), type='MADE_CONTRIBUTIONS_TO', properties={}), Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Node(id='mass–energy equivalence formula E = mc2', type='Formula', properties={}), type='FORMULATED', properties={}), Relationship(source=Node(id='Albert Einstein', type='Person', properties={}), target=Node(

In [93]:
neo4j_graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

# Push graph to Neo4j
neo4j_graph.add_graph_documents(graph_docs)


/tmp/ipykernel_7091/1816067205.py:1: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-neo4j package and should be used instead. To use it run `pip install -U :class:`~langchain-neo4j` and import as `from :class:`~langchain_neo4j import Neo4jGraph``.
  neo4j_graph = Neo4jGraph(


In [95]:
qa_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=neo4j_graph,
    verbose=True,
    allow_dangerous_requests = True
)


In [100]:
result = qa_chain.invoke({"query" : "who was albert einstein ?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (n:Person {name:"Albert Einstein"}) RETURN n;
Full Context:
[]

> Finished chain.


In [101]:
result

{'query': 'who was albert einstein ?', 'result': "I don't know the answer."}

In [99]:
result

{'query': 'when and where was albert einstein born?',
 'result': "I don't know the answer."}